# What is PL/SQL?

### PL/SQL = Procedural Language / Structured Query Language
### Its Oracle progrramming language that extends SQL by adding proggramming features such as variables, conditions, loops, exception handelling, procedures, functions, packages, and triggers.

In [1]:
path = r'E:\Study\Github\Sec\Scripts\DB Loading\connect_db.py'
exec(open(path, encoding='utf-8').read())

In [6]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


In [15]:
%%plsql
DECLARE
    v_msg VARCHAR2(100) := 'Brilliant! Your database setup is officially complete.';
BEGIN
    DBMS_OUTPUT.PUT_LINE(v_msg);
END;

Brilliant! Your database setup is officially complete.


In [1]:
path = r'E:\Study\Github\Sec\Scripts\DB Loading\connect_db.py'
exec(open(path, encoding='utf-8').read())

In [7]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


In [11]:
%%plsql
DECLARE
    -- Cursor to find all tables owned by the current active user
    CURSOR c_user_tables IS
        SELECT table_name 
        FROM user_tables 
        WHERE table_name IN ('CUSTOMER_ORDERS', 'ORDER_ITEMS', 'ORDERS', 'PRODUCTS', 'CUSTOMERS', 'STORES');
BEGIN
    DBMS_OUTPUT.PUT_LINE('🧹 Initializing standard table teardown process...');
    
    FOR r_tab IN c_user_tables LOOP
        BEGIN
            -- Execute clean drops individually to avoid CASCADE security rules
            EXECUTE IMMEDIATE 'DROP TABLE ' || r_tab.table_name;
            DBMS_OUTPUT.PUT_LINE('✅ Dropped table: ' || r_tab.table_name);
        EXCEPTION
            WHEN OTHERS THEN
                -- If a table has active foreign keys, drop it normally by clearing references first
                DBMS_OUTPUT.PUT_LINE('⚠️ standard drop failed for ' || r_tab.table_name || '. Retrying with constraint drop...');
                BEGIN
                    EXECUTE IMMEDIATE 'DROP TABLE ' || r_tab.table_name || ' CASCADE CONSTRAINTS';
                EXCEPTION
                    WHEN OTHERS THEN
                        DBMS_OUTPUT.PUT_LINE('❌ ORA Error on ' || r_tab.table_name || ': ' || SQLERRM);
                END;
        END;
    END LOOP;
    
    DBMS_OUTPUT.PUT_LINE('🎉 Cleanup phase evaluation completed.');
END;


🧹 Initializing standard table teardown process...
🎉 Cleanup phase evaluation completed.


In [12]:
%%plsql
DECLARE
    v_table_count NUMBER;
BEGIN
    SELECT COUNT(*) INTO v_table_count 
    FROM user_tables 
    WHERE table_name IN ('CUSTOMER_ORDERS', 'ORDER_ITEMS', 'ORDERS', 'PRODUCTS', 'CUSTOMERS', 'STORES');
    
    DBMS_OUTPUT.PUT_LINE('Active schema tables remaining: ' || v_table_count);
END;


Active schema tables remaining: 0


In [16]:
import os
import re
from IPython import get_ipython
import oracledb


def inject_downloaded_schema_safely():
    # 1. Point to your specific local script file
    script_path = r"E:\Study\Github\Repositories\Learn\SQL\PL SQL\PL SQL By Prashant\Sample Data\db-sample-schemas-23.3\customer_orders\co_install.sql"

    if not os.path.exists(script_path):
        print(f"❌ Script not found: {script_path}")
        return

    # 2. Extract active background notebook session details
    ip = get_ipython()
    if ip is None:
        return

    # 3. Retrieve the active registration module function from the global namespace
    plsql_magic = ip.magics_manager.magics['cell'].get('plsql')
    if plsql_magic is None:
        print("❌ Active Oracle database connection session not found. Please execute connect_oracle() first.")
        return

    # 4. Use Python introspection to securely pull the live engine cursor from your connect_db module
    try:
        closure_vars = plsql_magic.__closure__
        cursor = next(c.cell_contents for c in closure_vars if isinstance(
            c.cell_contents, oracledb.Cursor))
        connection = cursor.connection
    except Exception:
        print("❌ Could not bind direct session memory stream pipeline. Please refresh connect_oracle().")
        return

    print("⏳ Synchronizing direct cursor pipeline to Oracle cloud instance...")

    try:
        with open(script_path, "r", encoding="utf-8") as f:
            full_content = f.read()

        # Phase A: Strip out SQL*Plus client command-line instructions cleanly
        clean_lines = []
        for line in full_content.splitlines():
            line_strip = line.strip()
            if not line_strip or line_strip.upper().startswith(("SET ", "PROMPT ", "ACCEPT ", "SHOW ", "EXIT", "REM ")):
                continue
            # Handle inline comments
            line_clean = re.sub(r'--.*$', '', line)
            clean_lines.append(line_clean)

        processed_script = "\n".join(clean_lines)

        # Phase B: Cleanly segment standard SQL queries from full PL/SQL boundary blocks
        statements = []
        plsql_pattern = re.compile(
            r'(?:CREATE(?:\s+OR\s+REPLACE)?\s+(?:PROCEDURE|FUNCTION|TRIGGER|PACKAGE|VIEW)|DECLARE|BEGIN).*?END\s*;/|/',
            re.IGNORECASE | re.DOTALL
        )

        last_idx = 0
        for match in plsql_pattern.finditer(processed_script):
            start, end = match.span()
            before = processed_script[last_idx:start].strip()
            if before:
                for chunk in before.split(";"):
                    if chunk.strip():
                        statements.append((False, chunk.strip()))

            statements.append((True, match.group().strip()))
            last_idx = end

        rem = processed_script[last_idx:].strip()
        if rem:
            for chunk in rem.split(";"):
                if chunk.strip():
                    statements.append((False, chunk.strip()))

        # Phase C: Execute objects sequentially directly on the core database link
        print(
            f"📦 Executing {len(statements)} segmented schema instructions...")
        success_count = 0
        ignored_warnings = 0

        for is_plsql, stmt in statements:
            if stmt.endswith('/'):
                stmt = stmt[:-1].strip()
            if not is_plsql and stmt.endswith(';'):
                stmt = stmt[:-1].strip()
            if not stmt:
                continue

            try:
                if is_plsql:
                    cursor.callproc("dbms_output.enable")

                cursor.execute(stmt)
                success_count += 1
            except oracledb.DatabaseError as e:
                error, = e.args
                if error.code in (942, 1432, 2289, 4043):
                    ignored_warnings += 1
                else:
                    print(
                        f"⚠️ Query skipped: {stmt[:50]}... | Error: {error.message}")

        connection.commit()
        print("\n🎉 Parsing and schema construction complete!")
        print(f"✅ Successfully written data statements: {success_count}")
        print(
            f"🧹 Handled cleanup table warning indicators: {ignored_warnings}")

    except Exception as e:
        print(f"❌ Automation pipeline failed: {e}")


# Run the optimized execution engine stream
inject_downloaded_schema_safely()

⏳ Synchronizing direct cursor pipeline to Oracle cloud instance...
📦 Executing 29 segmented schema instructions...
⚠️ Query skipped: rem
rem
rem
rem
rem
rem
rem
rem
rem
rem
rem
rem 
r... | Error: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/
⚠️ Query skipped: END IF... | Error: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/
⚠️ Query skipped: END... | Error: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/
⚠️ Query skipped: COLUMN property_value NEW_VALUE var_default_tables... | Error: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/
⚠️ Query skipped: DECLARE
   v_tbs_exists   NUMBER := 0... | Error: ORA-06550: line 2, column 29:
PLS-00103: Encountered the symbol "end-of-file" when expecting one of the following:

   * & = - + ; < / > at in is mod remainder not rem
   <an exponent (**)> <> or != or ~= >= <= <> and or like like2


In [18]:
connect_oracle()

✨ Permanent %%plsql registered from secrets file! Session is live.


In [19]:
%%plsql
DECLARE
    CURSOR c_co_tables IS
        SELECT table_name 
        FROM user_tables 
        WHERE table_name IN ('CUSTOMER_ORDERS', 'ORDER_ITEMS', 'ORDERS', 'PRODUCTS', 'CUSTOMERS', 'STORES');
BEGIN
    FOR r_tab IN c_co_tables LOOP
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE ' || r_tab.table_name || ' CASCADE CONSTRAINTS';
            DBMS_OUTPUT.PUT_LINE('🧹 Cleared table: ' || r_tab.table_name);
        EXCEPTION
            WHEN OTHERS THEN
                NULL; -- Suppress if already dropped
        END;
    END LOOP;
    DBMS_OUTPUT.PUT_LINE('✅ Clean workspace ready for data injection.');
END;


✅ Clean workspace ready for data injection.


In [20]:
import os
import re
from IPython import get_ipython
import oracledb


def load_customer_orders_schema():
    # 1. Base directory where your sample scripts are located
    base_dir = r"E:\Study\Github\Repositories\Learn\SQL\PL SQL\PL SQL By Prashant\Sample Data\db-sample-schemas-23.3\customer_orders"

    # 2. Target the actual data layout and data files directly, skipping co_install.sql
    target_files = [
        os.path.join(base_dir, "co_create.sql"),
        os.path.join(base_dir, "co_populate.sql")
    ]

    # Retrieve notebook context
    ip = get_ipython()
    if ip is None:
        return

    # Securely retrieve the active underlying driver connection cursor from memory
    plsql_magic = ip.magics_manager.magics['cell'].get('plsql')
    if plsql_magic is None:
        print("❌ Active session not found. Please run connect_oracle() first.")
        return

    try:
        closure_vars = plsql_magic.__closure__
        cursor = next(c.cell_contents for c in closure_vars if isinstance(
            c.cell_contents, oracledb.Cursor))
        connection = cursor.connection
    except Exception:
        print("❌ Memory pipeline binding failed. Please run connect_oracle() again.")
        return

    print("⏳ Starting clean schema execution directly on live database stream...")

    for file_path in target_files:
        if not os.path.exists(file_path):
            print(f"❌ Target script file is missing: {file_path}")
            return

        print(f"📦 Processing: {os.path.basename(file_path)}...")

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        # Clean out comment headers and command-line noise lines
        clean_lines = []
        for line in content.splitlines():
            line_strip = line.strip()
            if not line_strip or line_strip.upper().startswith(("SET ", "PROMPT ", "ACCEPT ", "SHOW ", "EXIT", "REM ")):
                continue
            # Remove inline SQL comments
            line_clean = re.sub(r'--.*$', '', line)
            clean_lines.append(line_clean)

        processed_script = "\n".join(clean_lines)

        # Segment by semicolon safely (the tables/inserts are simple standard SQL statements)
        statements = processed_script.split(";")

        success_count = 0
        warning_count = 0

        for stmt in statements:
            stmt_clean = stmt.strip()
            # Trim lingering client execution flags
            if stmt_clean.endswith('/'):
                stmt_clean = stmt_clean[:-1].strip()
            if not stmt_clean:
                continue

            try:
                cursor.execute(stmt_clean)
                success_count += 1
            except oracledb.DatabaseError as e:
                error, = e.args
                # Silently catch standard initialization table dropping errors
                if error.code in (942, 1432, 2289, 4043):
                    warning_count += 1
                else:
                    print(
                        f"⚠️ Statement skipped: {stmt_clean[:40]}... | Error: {error.message}")

        connection.commit()
        print(
            f"✅ Successfully written items: {success_count} | 🧹 Handled initialization drop messages: {warning_count}\n")

    print("🎉 All Customer Orders data sets successfully injected into your cloud schema!")


# Run the ingestion script
load_customer_orders_schema()

⏳ Starting clean schema execution directly on live database stream...
📦 Processing: co_create.sql...
⚠️ Statement skipped: rem
rem
rem
rem
rem
rem
rem
rem
rem
rem
... | Error: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/
✅ Successfully written items: 94 | 🧹 Handled initialization drop messages: 25

📦 Processing: co_populate.sql...
⚠️ Statement skipped: rem
rem
rem
rem
rem
rem
rem
rem
rem
rem
... | Error: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/
⚠️ Statement skipped: BEGIN
  INSERT INTO customers (customer_... | Error: ORA-06550: line 2, column 115:
PLS-00103: Encountered the symbol "end-of-file" when expecting one of the following:

   ;
Help: https://docs.oracle.com/error-help/db/ora-06550/
⚠️ Statement skipped: END... | Error: ORA-00900: invalid SQL statement
Help: https://docs.oracle.com/error-help/db/ora-00900/
⚠️ Statement skipped: /
DECLARE
  prod_details VARCHAR2(32767)... | Error: ORA-0090

KeyboardInterrupt: 

In [21]:
%%plsql
DECLARE
    v_rows NUMBER;
BEGIN
    SELECT COUNT(*) INTO v_rows FROM orders;
    DBMS_OUTPUT.PUT_LINE('📊 Complete Ingestion Success! Active dataset records found inside the orders table: ' || v_rows);
END;


📊 Complete Ingestion Success! Active dataset records found inside the orders table: 2


In [22]:
%%plsql
DECLARE
    CURSOR c_co_tables IS
        SELECT table_name 
        FROM user_tables 
        WHERE table_name IN ('CUSTOMER_ORDERS', 'ORDER_ITEMS', 'ORDERS', 'PRODUCTS', 'CUSTOMERS', 'STORES');
BEGIN
    FOR r_tab IN c_co_tables LOOP
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE ' || r_tab.table_name || ' CASCADE CONSTRAINTS';
            DBMS_OUTPUT.PUT_LINE('🧹 Dropped fragmented object: ' || r_tab.table_name);
        EXCEPTION
            WHEN OTHERS THEN NULL;
        END;
    END LOOP;
    DBMS_OUTPUT.PUT_LINE('✅ Environment successfully reset.');
END;


🧹 Dropped fragmented object: ORDERS
🧹 Dropped fragmented object: ORDER_ITEMS
🧹 Dropped fragmented object: PRODUCTS
🧹 Dropped fragmented object: STORES
✅ Environment successfully reset.


In [23]:
import os
import re
from IPython import get_ipython
import oracledb


def load_customer_orders_schema_properly():
    base_dir = r"E:\Study\Github\Repositories\Learn\SQL\PL SQL\PL SQL By Prashant\Sample Data\db-sample-schemas-23.3\customer_orders"

    file_create = os.path.join(base_dir, "co_create.sql")
    file_populate = os.path.join(base_dir, "co_populate.sql")

    ip = get_ipython()
    if ip is None:
        return

    # Direct extraction of cursor from active session memory
    plsql_magic = ip.magics_manager.magics['cell'].get('plsql')
    if plsql_magic is None:
        print("❌ Session dropped. Please execute connect_oracle() first.")
        return

    try:
        closure_vars = plsql_magic.__closure__
        cursor = next(c.cell_contents for c in closure_vars if isinstance(
            c.cell_contents, oracledb.Cursor))
        connection = cursor.connection
    except Exception:
        print("❌ Memory context binding failure. Re-execute connect_oracle().")
        return

    print("⏳ Initializing database cursor pipeline...")

    # --- PHASE 1: EXECUTE STRUCTURES (co_create.sql) ---
    if os.path.exists(file_create):
        print("📦 Processing Structure: co_create.sql...")
        with open(file_create, "r", encoding="utf-8") as f:
            content = f.read()

        # Strip out terminal script controls
        clean_lines = [l for l in content.splitlines() if not l.strip().upper(
        ).startswith(("SET ", "PROMPT ", "ACCEPT ", "SHOW ", "EXIT", "REM "))]
        processed = "\n".join(clean_lines)

        # Tables can be cleanly isolated via standard semicolon mapping
        statements = [s.strip() for s in processed.split(";") if s.strip()]

        success_count = 0
        for stmt in statements:
            if stmt.endswith('/'):
                stmt = stmt[:-1].strip()
            if not stmt:
                continue
            try:
                cursor.execute(stmt)
                success_count += 1
            except oracledb.DatabaseError as e:
                error, = e.args
                # Suppress normal initialization drop warnings
                if error.code not in (942, 1432, 2289):
                    print(
                        f"⚠️ skipped structure line: {stmt[:40]}... | Error: {error.message}")

        connection.commit()
        print(
            f"✅ Successfully initialized {success_count} structural table allocations.")

    # --- PHASE 2: EXECUTE BLOCKS (co_populate.sql) ---
    if os.path.exists(file_populate):
        print("\n📦 Processing Content Blocks: co_populate.sql...")
        with open(file_populate, "r", encoding="utf-8") as f:
            content = f.read()

        # Clean comments and client command flags while preserving structural indentation blocks
        clean_lines = []
        for line in content.splitlines():
            line_strip = line.strip()
            if line_strip.upper().startswith(("SET ", "PROMPT ", "ACCEPT ", "SHOW ", "EXIT", "REM ")):
                continue
            # Keep structural forward slashes intact but strip commented noise lines
            line_clean = re.sub(r'--.*$', '', line)
            clean_lines.append(line_clean)

        processed_data = "\n".join(clean_lines)

        # CRITICAL FIX: Split using Oracle's native batch termination boundary flag (\n/\n)
        # instead of stripping individual semicolons out of the nested code loops
        blocks = re.split(r'\n\s*/\s*\n', processed_data)

        success_blocks = 0
        print("⏳ Injecting multi-line row execution arrays into cloud instance...")

        for block in blocks:
            block_clean = block.strip()
            if not block_clean:
                continue

            try:
                # Fire the complete, unbroken procedural chunk directly into the database engine
                cursor.execute(block_clean)
                success_blocks += 1
            except oracledb.DatabaseError as e:
                error, = e.args
                print(
                    f"❌ Block Ingestion Aborted: {block_clean[:80]}... \n↳ Error: {error.message}\n")

        connection.commit()
        print(
            f"✅ Successfully written transactional rows: {success_blocks} data arrays.")
        print("🎉 Database optimization complete. All records populated safely!")


load_customer_orders_schema_properly()

⏳ Initializing database cursor pipeline...
📦 Processing Structure: co_create.sql...
⚠️ skipped structure line: rem
rem
rem
rem
rem
rem
rem
rem
rem
rem
... | Error: DPY-4011: the database or network closed the connection
Help: https://python-oracledb.readthedocs.io/en/latest/user_guide/troubleshooting.html#dpy-4011
⚠️ skipped structure line: CREATE TABLE stores
(
  store_id        ... | Error: DPY-4011: the database or network closed the connection
Help: https://python-oracledb.readthedocs.io/en/latest/user_guide/troubleshooting.html#dpy-4011
⚠️ skipped structure line: CREATE TABLE products
(
  product_id    ... | Error: DPY-4011: the database or network closed the connection
Help: https://python-oracledb.readthedocs.io/en/latest/user_guide/troubleshooting.html#dpy-4011
⚠️ skipped structure line: CREATE TABLE orders
(
  order_id       I... | Error: DPY-4011: the database or network closed the connection
Help: https://python-oracledb.readthedocs.io/en/latest/user_guide/troubleshooting.ht

DatabaseError: DPY-4011: the database or network closed the connection
Help: https://python-oracledb.readthedocs.io/en/latest/user_guide/troubleshooting.html#dpy-4011